# Report figures and tables
FINS5566 group project: what drives bid-ask spreads on Nasdaq Copenhagen, informed trading or volatility?

This notebook only reads the outputs of the pipeline (`analysis/01`–`08`) and turns them into the
final report figures (`outputs/figures/report_*.png`) and tables (`outputs/tables/report_*.csv`).
Run the numbered scripts first. Sample: 6 stocks, 1–30 Oct 2026; in-sample 1–21 Oct, out-of-sample 22–30 Oct.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from tifs_group_project import config

T, F = config.TABLES, config.FIGURES
STOCKS = list(config.STOCKS)
COLORS = dict(
    zip(STOCKS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"])
)
GROUP_COLORS = {"constrained": "#2a78d6", "unconstrained": "#eb6834"}
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e5e4e0",
        "grid.linewidth": 0.6,
        "lines.linewidth": 1.6,
        "figure.dpi": 110,
        "savefig.dpi": 200,
        "font.size": 9,
    }
)


def save(fig, name: str) -> None:
    fig.savefig(F / f"report_{name}.png", bbox_inches="tight")


def group_of(stock: str) -> str:
    return "constrained" if stock in config.CONSTRAINED else "unconstrained"

## Table 1. Data summary
Continuous lit trades (09:00–16:55), with the tick constraint that splits the sample into two groups.

In [ ]:
data = pd.read_csv(T / "01_data_summary.csv", index_col="stock")
sign = pd.read_csv(T / "02_signing_summary.csv", index_col="stock")
tick = pd.read_csv(T / "03_tick_constraint.csv", index_col="stock")
table1 = pd.DataFrame(
    {
        "Group": [group_of(s) for s in STOCKS],
        "Lit trades per day": data["lit_trades_per_day"].round(0).astype(int),
        "Turnover (DKK m/day)": data["turnover_dkk_m_per_day"].round(0).astype(int),
        "Fills per order": sign["fills_per_order"],
        "Tick (bps)": tick["tick_bps"].round(1),
        "Median spread (bps)": tick["median_spread_bps"].round(1),
        "Time at 1 tick (%)": tick["pct_time_1tick"].round(0).astype(int),
        "Signed by quote rule (%)": sign["pct_quote_rule"],
    },
    index=STOCKS,
)
table1.index.name = "Stock"
table1.to_csv(T / "report_table1_data_summary.csv")
display(table1)

## Figure 1. Intraday patterns and the tick constraint
Spread, turnover and volatility by minute (in-sample days, indexed to each stock's daily mean, 5-min smoothed),
and the share of time the spread sits at one tick.

In [ ]:
intra = pd.read_csv(T / "03_intraday_by_minute.csv")
b5 = pd.read_parquet(config.PROCESSED / "bars_5min.parquet")
b5 = b5[b5["sample"] == "IS"]
b5["half_hour"] = b5["minute_of_day"] // 30 * 30
tick_share = b5.groupby(["stock", "half_hour"]).apply(
    lambda x: (
        np.average(x["pct_time_1tick"].fillna(0), weights=x["q_sec"])
        if x["q_sec"].sum() > 0
        else np.nan
    ),
    include_groups=False,
)

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True)
panels = [
    ("spread_bps_tw", "Quoted spread (time-weighted)"),
    ("dkk_turnover", "Turnover"),
    ("vol_1min_bps", "Volatility (RMS 1-min mid return)"),
]
for ax, (col, title) in zip(axes.flat, panels):
    for s in STOCKS:
        x = intra[intra["stock"] == s].set_index("mod")[col]
        x = (x / x.mean()).rolling(5, center=True, min_periods=1).mean()
        ax.plot(x.index, x.values, color=COLORS[s], label=s, linewidth=1.2)
    ax.axhline(1, color="#8a8984", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("× stock's daily mean")
ax = axes[1, 1]
for s in STOCKS:
    x = tick_share.loc[s]
    ax.plot(
        x.index + 15,
        x.values,
        color=COLORS[s],
        marker="o",
        markersize=3,
        label=s,
        linestyle="-" if s in config.CONSTRAINED else "--",
    )
ax.set_ylim(0, 100)
ax.set_title("Time with spread = 1 tick (%; solid = constrained group)", loc="left")
ticks = np.arange(0, 481, 60)
for a in axes[1]:
    a.set_xticks(ticks, [f"{9 + t // 60:02d}:00" for t in ticks])
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    ncol=6,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
save(fig, "fig1_intraday_tick")
plt.show()

## Figure 2. Price impact by stock
Permanent impact of a buy order from the Hasbrouck (1991) VAR (20-order cumulative IRF), against the
spread decomposition at +5 min and the effective half-spread (medians across days / rolling windows).

In [ ]:
imp = pd.read_csv(T / "05_impact_summary.csv", index_col="stock").reindex(STOCKS)
order = imp.sort_values("eff_hs_med").index
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(order))
ax.hlines(y, 0, imp.loc[order, "eff_hs_med"], color="#e5e4e0", linewidth=6)
ax.scatter(
    imp.loc[order, "eff_hs_med"],
    y,
    color="#8a8984",
    s=60,
    label="Effective half-spread",
    zorder=3,
)
ax.scatter(
    imp.loc[order, "impact_5m_med"],
    y,
    color="#eb6834",
    s=60,
    marker="s",
    label="Price impact at +5 min (decomposition)",
    zorder=3,
)
ax.scatter(
    imp.loc[order, "var_impact_med"],
    y,
    color="#2a78d6",
    s=70,
    marker="D",
    label="Permanent impact (Hasbrouck VAR)",
    zorder=4,
)
ax.set_yticks(y, [f"{s} ({'C' if s in config.CONSTRAINED else 'U'})" for s in order])
ax.set_xlabel("bps")
ax.set_xlim(0, None)
ax.set_title(
    "Adverse selection takes most of the effective spread (C = tick-constrained)",
    loc="left",
)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
save(fig, "fig2_impact_by_stock")
plt.show()
display(
    imp[
        [
            "group",
            "var_impact_med",
            "impact_5m_med",
            "eff_hs_med",
            "realised_5m_med",
            "pct_eff_spread_adverse_sel",
        ]
    ].round(2)
)

## Table 2. Spread regressions by group (in-sample, 1–21 Oct)
Dependent variable: quoted half-spread (bps) per 5-min bar. Cells: coefficient [Newey–West t] and the effect of a
one within-stock SD increase (bps). All specifications include lagged trades, turnover and |order imbalance|.
Original = shared time-of-day dummies + stock FE (script 07); refined = stock × time buckets (script 08).

In [ ]:
coef = pd.read_csv(T / "08_r1_coefficients.csv")
coef = coef[coef["spec"].isin(["original", "refined"])]
names = {
    "impact_roll_ewm_bps": "Impact, rolling (last 300 orders)",
    "impact_prevday_bps": "Impact, previous day",
    "log_vol_har": "Vol forecast, log(HAR / seasonal)",
}
rows = []
for (grp, spec), d in coef.groupby(["group", "spec"], sort=False):
    d = d.set_index("var")
    for v, label in names.items():
        r = d.loc[v]
        star = (
            "***"
            if r["p"] < 0.01
            else "**"
            if r["p"] < 0.05
            else "*"
            if r["p"] < 0.1
            else ""
        )
        rows.append(
            {
                "Group": grp,
                "Spec": spec,
                "Driver": label,
                "Coef [t]": f"{r['coef']:.3f}{star} [{r['t']:.2f}]",
                "Effect of +1 SD (bps)": round(r["effect_1sd"], 3),
            }
        )
    rows.append(
        {
            "Group": grp,
            "Spec": spec,
            "Driver": "N / R²",
            "Coef [t]": f"{int(d['n'].iloc[0])} / {d['r2'].iloc[0]:.3f}",
            "Effect of +1 SD (bps)": np.nan,
        }
    )
table2 = pd.DataFrame(rows).pivot_table(
    index=["Group", "Driver"],
    columns="Spec",
    values=["Coef [t]", "Effect of +1 SD (bps)"],
    aggfunc="first",
    sort=False,
)
table2.to_csv(T / "report_table2_spread_regressions.csv")
display(table2)

## Figure 3 and Table 3. Out-of-sample race (22–30 Oct)
RMSE of the quoted half-spread per 5-min bar. Benchmarks: in-sample average, yesterday's average, last bar's spread.
Out-of-sample use: script 07 was the first evaluation, refinement 1 the second, LightGBM the third.
Diebold–Mariano statistics (negative = model beats the benchmark): group loss averaged across stocks per bar.

In [ ]:
rmse = (
    pd.read_csv(T / "08_oos_rmse.csv")
    .query("subset == 'all OOS days'")
    .set_index("unit")
)
dm = pd.read_csv(T / "08_oos_diebold_mariano.csv").query("subset == 'all OOS days'")
race = [
    "IS average",
    "yesterday",
    "last bar",
    "original: combined",
    "refined: combined",
    "log shared: combined",
    "refined + lagged spread",
    "LightGBM",
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, grp in zip(axes, GROUP_COLORS):
    vals = rmse.loc[grp, race]
    colors = ["#c3c2b7"] * 3 + [GROUP_COLORS[grp]] * 5
    ax.barh(np.arange(len(race))[::-1], vals, color=colors, height=0.65)
    for k, v in enumerate(vals):
        ax.text(v + 0.01, len(race) - 1 - k, f"{v:.2f}", va="center", fontsize=8)
    ax.set_yticks(np.arange(len(race))[::-1], race)
    ax.set_xlabel("RMSE (bps)")
    ax.set_title(f"{grp.capitalize()} group", loc="left")
    ax.set_xlim(0, vals.max() * 1.15)
fig.suptitle(
    "Out-of-sample RMSE of the quoted half-spread (grey = benchmarks)",
    x=0.01,
    ha="left",
)
fig.tight_layout()
save(fig, "fig3_oos_race")
plt.show()

units = [*GROUP_COLORS, *STOCKS]
table3 = rmse.loc[units, race].round(3)
for model in ["refined: combined", "log shared: combined", "LightGBM"]:
    for bench in ["yesterday", "last bar"]:
        d = dm[(dm["model"] == model) & (dm["vs"] == bench)].set_index("unit")
        if d.empty:  # pair not tested in script 08
            continue
        table3[f"DM {model} vs {bench}"] = d.loc[units, "dm_stat"].round(2)
table3.to_csv(T / "report_table3_oos_race.csv")
display(table3)

## Figure 4. What the LightGBM forecast relies on (SHAP)
Mean |SHAP| on in-sample rows, by group: the persistence of the spread and the stock dominate; among the
economic drivers, impact matters more than the volatility forecast.

In [ ]:
shap_imp = pd.read_csv(T / "08_r2_shap_importance.csv", index_col=0)
labels = {
    "lag_quoted_hs": "Spread, last bar",
    "stock": "Stock",
    "minute_of_day": "Minute of day",
    "impact_prevday_bps": "Impact, previous day",
    "impact_roll_ewm_bps": "Impact, rolling",
    "min_since_us_open": "Minutes since US open",
    "log_vol_har": "Vol forecast (HAR)",
    "bucket": "Time bucket",
    "lag_log_trades": "Trades, last bar",
    "lag_log_turnover": "Turnover, last bar",
    "lag_ofi": "Order imbalance, last bar",
    "lag_abs_ofi": "|Order imbalance|, last bar",
}
s = shap_imp.sort_values("unconstrained")
fig, ax = plt.subplots(figsize=(7, 4.2))
y = np.arange(len(s))
for k, grp in enumerate(GROUP_COLORS):
    ax.barh(
        y + (k - 0.5) * 0.38, s[grp], height=0.36, color=GROUP_COLORS[grp], label=grp
    )
ax.set_yticks(y, [labels.get(i, i) for i in s.index])
ax.set_xlabel("Mean |SHAP| (bps of predicted half-spread)")
ax.set_title("LightGBM feature reliance by group", loc="left")
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
save(fig, "fig4_shap")
plt.show()

## Figure 5. Drivers by volatility regime (HMM)
2-state Gaussian HMM on deseasonalised 5-min realised volatility, fitted in-sample, forward-filtered (no look-ahead).
Coefficients (95% Newey–West CI) from the refined specification with drivers interacted with the stressed regime.

In [ ]:
reg = pd.read_csv(T / "08_hmm_drivers_by_regime.csv").query("hmm == 'vol only (main)'")
shares = pd.read_csv(T / "08_hmm_stressed_share.csv").query("hmm == 'vol only (main)'")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, grp in zip(axes, GROUP_COLORS):
    d = reg[reg["group"] == grp].set_index("var").loc[list(names)]
    y = np.arange(len(d))[::-1]
    for off, regime, marker, color in [
        (0.12, "calm", "o", "#8a8984"),
        (-0.12, "stressed", "D", GROUP_COLORS[grp]),
    ]:
        c, t = d[f"{regime}_coef"], d[f"{regime}_t"]
        ax.errorbar(
            c,
            y + off,
            xerr=1.96 * (c / t).abs(),
            fmt=marker,
            color=color,
            ecolor=color,
            capsize=3,
            markersize=6,
            label=regime,
        )
    ax.axvline(0, color="#52514e", linewidth=0.8)
    ax.set_yticks(y, [names[v] for v in d.index])
    ax.set_xlabel("Coefficient (bps per unit)")
    ax.set_title(f"{grp.capitalize()} group", loc="left")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(
    "Volatility widens tick-constrained spreads only in the stressed regime",
    x=0.01,
    ha="left",
)
fig.tight_layout()
save(fig, "fig5_hmm_regimes")
plt.show()

table5 = reg[
    [
        "group",
        "var",
        "calm_coef",
        "calm_t",
        "stressed_coef",
        "stressed_t",
        "diff_t",
        "pct_bars_stressed",
    ]
]
table5.round(3).to_csv(T / "report_table5_hmm_regimes.csv", index=False)
display(table5.round(3))
display(
    shares.dropna(subset=["pct_stressed_event_day"])[
        ["stock", "pct_stressed_event_day", "pct_stressed_IS_other_days"]
    ].round(1)
)